# Parcial 1 — Programación para Ciencia de Datos (SCY1101)

**Dataset:** `urgencias_noprocesados_grupo06.csv`  
**Objetivo:** Resolver el encargo completo de la rúbrica: manipulación, limpieza, transformaciones avanzadas, validación e informe con visualizaciones.

## 1) Plan de trabajo alineado a la rúbrica (Encargo)

1. **Manipulación en Pandas:** filtros, agrupaciones múltiples y joins.
2. **Reporte y visualización:** métricas + gráficos claros.
3. **Transformaciones avanzadas:** vectorización, broadcasting, pivot/reshape, chunking.
4. **Flujo de limpieza:** estandarización, imputación, deduplicación y consistencia.
5. **Entorno reproducible:** evidencia de versión de librerías y `requirements.txt`.
6. **Integridad/procedencia:** hash de archivo, validación de esquema y reglas de calidad.

In [1]:
from __future__ import annotations

import hashlib
from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

def detect_project_root() -> Path:
    """Detecta raíz del proyecto aunque el notebook se ejecute desde /notebooks."""
    candidates = [Path.cwd(), Path.cwd().parent]
    for base in candidates:
        if (base / 'urgencias_noprocesados_grupo06.csv').exists() and (base / 'EV PARCIAL 1 SCY1101_ESTUDIANTE.pdf').exists():
            return base.resolve()
    raise FileNotFoundError('No se encontró la raíz del proyecto (faltan CSV o PDF).')

PROJECT_ROOT = detect_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

for directory in [DATA_DIR, OUTPUTS_DIR, NOTEBOOKS_DIR]:
    directory.mkdir(exist_ok=True)

# Entrada preferente: raíz; fallback: data/
raw_candidates = [
    PROJECT_ROOT / 'urgencias_noprocesados_grupo06.csv',
    DATA_DIR / 'raw' / 'urgencias_noprocesados_grupo06.csv',
    DATA_DIR / 'urgencias_noprocesados_grupo06.csv',
]
RAW_PATH = next((p for p in raw_candidates if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError('No se encontró urgencias_noprocesados_grupo06.csv en rutas esperadas.')

PROCESSED_PATH = DATA_DIR / 'urgencias_procesados_grupo06.csv'

print('Directorio de trabajo actual:', Path.cwd().resolve())
print('PROJECT_ROOT detectado:', PROJECT_ROOT)
print('Archivo fuente:', RAW_PATH)
print('Archivo fuente existe:', RAW_PATH.exists())


Directorio de trabajo actual: /Users/benja/Documents/Duoc/5to Semestre/Programacion Ciencia de Datos/EA1/Parcial 1/notebooks
PROJECT_ROOT detectado: /Users/benja/Documents/Duoc/5to Semestre/Programacion Ciencia de Datos/EA1/Parcial 1
Archivo fuente: /Users/benja/Documents/Duoc/5to Semestre/Programacion Ciencia de Datos/EA1/Parcial 1/urgencias_noprocesados_grupo06.csv
Archivo fuente existe: True


In [2]:
def sha256_file(path: Path) -> str:
    """Calcula hash SHA256 de un archivo para control de procedencia."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


def parse_mixed_date(value: str):
    """Parsea fechas en múltiples formatos comunes del dataset."""
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()
    fmts = ["%d.%m.%y", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y", "%d-%m-%y"]

    for fmt in fmts:
        try:
            return pd.Timestamp(datetime.strptime(value, fmt).date())
        except ValueError:
            continue

    return pd.to_datetime(value, errors='coerce', dayfirst=True)


def mode_or_nan(series: pd.Series):
    """Retorna la moda de una serie o NaN si está vacía."""
    m = series.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

In [3]:
raw_hash = sha256_file(RAW_PATH)

try:
    df_raw = pd.read_csv(RAW_PATH)
except Exception as e:
    raise RuntimeError(f"No se pudo leer el CSV: {e}")

print(f"Filas x columnas: {df_raw.shape}")
print(f"SHA256 archivo original: {raw_hash}")
df_raw.head(3)

Filas x columnas: (4742, 29)
SHA256 archivo original: 9e149a01372933d09829bab46bdce7418c009e518b2d3d887efdbd08b297555f


,EstablecimientoCodigo,EstablecimientoGlosa,RegionCodigo,RegionGlosa,ComunaCodigo,ComunaGlosa,ServicioSaludCodigo,ServicioSaludGlosa,TipoEstablecimiento,DependenciaAdministrativa,NivelAtencion,TipoUrgencia,Latitud,Longitud,NivelComplejidad,Anio,SemanaEstadistica,OrdenCausa,Causa,NumTotal,NumMenor1Anio,Num1a4Anios,Num5a14Anios,Num15a64Anios,Num65oMas,FechaAtencionTexto,SexoPaciente,PrioridadTriage,CostoAtencionCLP
0,110901,Hospital Regional de Arica Dr. Juan Noe,15,Arica y Parinacota,15101,Arica,1,Servicio de Salud Arica,Hospital,Servicio de Salud,Terciario,Hospitalaria (UEH),-18.4783,-70.3126,Alta Complejidad,2023,1,1,Influenza y neumonia,201,19,21,27,87,47,02.01.23,F,C5,87482.0
1,110901,Hospital Regional de Arica Dr. Juan Noe,15,Arica y Parinacota,15101,Arica,1,Servicio de Salud Arica,Hospital,Servicio de Salud,Terciario,Hospitalaria (UEH),-18.4783,-70.3126,Alta Complejidad,2023,1,2,Infeccion respiratoria aguda alta,193,14,25,30,106,18,02.01.23,F,C4,105598.0
2,110901,Hospital Regional de Arica Dr. Juan Noe,15,Arica y Parinacota,15101,Arica,1,Servicio de Salud Arica,Hospital,Servicio de Salud,Terciario,Hospitalaria (UEH),-18.4783,-70.3126,Alta Complejidad,2023,1,3,Bronquitis/Bronquiolitis aguda,94,18,19,23,588,6,02-01-2023,F,C2,135016.0


In [4]:
print('Tipos de datos:')
print(df_raw.dtypes)
print()
print('Nulos por columna (top 15):')
print(df_raw.isna().sum().sort_values(ascending=False).head(15))
print()
print('Duplicados exactos:', df_raw.duplicated().sum())

age_cols = ['NumMenor1Anio', 'Num1a4Anios', 'Num5a14Anios', 'Num15a64Anios', 'Num65oMas']
sum_age = df_raw[age_cols].sum(axis=1)
print('Registros donde NumTotal != suma por tramos etarios:', (df_raw['NumTotal'] != sum_age).sum())

Tipos de datos:
EstablecimientoCodigo          int64
EstablecimientoGlosa             str
RegionCodigo                   int64
RegionGlosa                      str
ComunaCodigo                   int64
ComunaGlosa                      str
ServicioSaludCodigo            int64
ServicioSaludGlosa               str
TipoEstablecimiento              str
DependenciaAdministrativa        str
NivelAtencion                    str
TipoUrgencia                     str
Latitud                      float64
Longitud                     float64
NivelComplejidad                 str
Anio                           int64
SemanaEstadistica              int64
OrdenCausa                     int64
Causa                            str
NumTotal                       int64
NumMenor1Anio                  int64
Num1a4Anios                    int64
Num5a14Anios                   int64
Num15a64Anios                  int64
Num65oMas                      int64
FechaAtencionTexto               str
SexoPaciente          

## 2) Limpieza de datos con justificación técnica

Decisiones aplicadas:

- **Deduplicación**: se eliminan duplicados exactos para evitar doble conteo.
- **Estandarización de categorías**: `SexoPaciente` y `PrioridadTriage` tenían formatos inconsistentes.
- **Fechas mixtas**: normalización a `datetime` con parser multi-formato.
- **Imputación dirigida**:
  - Coordenadas por mediana del establecimiento.
  - Variables categóricas por moda del establecimiento.
  - Costo por mediana de prioridad de triage (fallback global).
- **Consistencia interna**: se recalcula `NumTotal` como suma de tramos etarios para eliminar contradicciones.
- **Outliers de costo**: winsorización p1-p99 para reducir impacto de extremos sin borrar casos.

In [ ]:
df = df_raw.copy()

df = df.drop_duplicates().copy()

text_cols = [
    'EstablecimientoGlosa', 'RegionGlosa', 'ComunaGlosa', 'ServicioSaludGlosa',
    'TipoEstablecimiento', 'DependenciaAdministrativa', 'NivelAtencion',
    'TipoUrgencia', 'NivelComplejidad', 'Causa', 'SexoPaciente', 'PrioridadTriage'
]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip()

sexo_map = {
    'F': 'F', 'f': 'F', 'femenino': 'F', 'Femenino': 'F',
    'M': 'M', 'm': 'M', 'masculino': 'M', 'Masculino': 'M'
}
df['SexoPaciente'] = df['SexoPaciente'].replace(sexo_map)
df['SexoPaciente'] = df['SexoPaciente'].where(df['SexoPaciente'].isin(['F', 'M']), 'ND')

triage_clean = (
    df['PrioridadTriage']
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)
triage_clean = triage_clean.where(triage_clean.isin(['C1', 'C2', 'C3', 'C4', 'C5']), 'ND')
df['PrioridadTriage'] = triage_clean

df['FechaAtencion'] = df['FechaAtencionTexto'].apply(parse_mixed_date)

for col in ['Latitud', 'Longitud']:
    grp_med = df.groupby('EstablecimientoCodigo')[col].transform('median')
    df[col] = df[col].fillna(grp_med)
    df[col] = df[col].fillna(df[col].median())

for col in ['DependenciaAdministrativa', 'TipoUrgencia', 'NivelComplejidad']:
    fill_by_est = df.groupby('EstablecimientoCodigo')[col].transform(mode_or_nan)
    df[col] = df[col].fillna(fill_by_est)
    df[col] = df[col].fillna('NoInformado')

cost_med_triage = df.groupby('PrioridadTriage')['CostoAtencionCLP'].transform('median')
df['CostoAtencionCLP'] = df['CostoAtencionCLP'].fillna(cost_med_triage)
df['CostoAtencionCLP'] = df['CostoAtencionCLP'].fillna(df['CostoAtencionCLP'].median())

df['NumTotalOriginal'] = df['NumTotal']
df['NumTotal'] = df[age_cols].sum(axis=1)
df['FlagTotalCorregido'] = df['NumTotalOriginal'] != df['NumTotal']

q1, q99 = df['CostoAtencionCLP'].quantile([0.01, 0.99])
df['CostoAtencionCLP'] = df['CostoAtencionCLP'].clip(lower=q1, upper=q99)

print('Registros tras limpieza:', len(df))
print('Nulos remanentes top 10:')
print(df.isna().sum().sort_values(ascending=False).head(10))

## 3) Manipulación en Pandas (filtros, agrupaciones y joins)

In [ ]:
filtro = (
    (df['Anio'] >= 2024) &
    (df['TipoEstablecimiento'].str.contains('Hospital', case=False, na=False)) &
    (df['PrioridadTriage'].isin(['C1', 'C2', 'C3']))
)

df_filtro = df.loc[filtro].copy()
print('Registros filtrados:', df_filtro.shape)

In [ ]:
agg_multi = (
    df_filtro
    .groupby(['RegionGlosa', 'SexoPaciente', 'PrioridadTriage'], as_index=False)
    .agg(
        atenciones_totales=('NumTotal', 'sum'),
        costo_total=('CostoAtencionCLP', 'sum'),
        costo_promedio=('CostoAtencionCLP', 'mean'),
        n_registros=('NumTotal', 'size')
    )
    .sort_values(['atenciones_totales', 'costo_total'], ascending=False)
)

agg_multi.head(10)

In [ ]:
macrozona_map = {
    'Arica y Parinacota': 'Norte',
    'Tarapaca': 'Norte',
    'Antofagasta': 'Norte',
    'Atacama': 'Norte',
    'Coquimbo': 'Norte Chico',
    'Valparaiso': 'Centro',
    'Metropolitana de Santiago': 'Centro',
    "Libertador General Bernardo O'Higgins": 'Centro Sur',
    'Maule': 'Centro Sur',
    'Nuble': 'Sur',
    'Biobio': 'Sur',
    'La Araucania': 'Sur',
    'Los Rios': 'Sur Austral',
    'Los Lagos': 'Sur Austral',
    'Aysen del General Carlos Ibanez del Campo': 'Austral',
    'Magallanes y de la Antartica Chilena': 'Austral'
}

dim_region = (
    df[['RegionCodigo', 'RegionGlosa']]
    .drop_duplicates()
    .assign(Macrozona=lambda x: x['RegionGlosa'].map(macrozona_map).fillna('NoClasificada'))
)

dim_calendar = (
    df[['Anio', 'SemanaEstadistica']]
    .drop_duplicates()
    .assign(
        TrimestreAprox=lambda x: pd.cut(
            x['SemanaEstadistica'],
            bins=[0, 13, 26, 39, 53],
            labels=['T1', 'T2', 'T3', 'T4'],
            include_lowest=True
        )
    )
)

df_enriched = (
    df
    .merge(dim_region, on=['RegionCodigo', 'RegionGlosa'], how='left')
    .merge(dim_calendar, on=['Anio', 'SemanaEstadistica'], how='left')
)

df_enriched[['RegionGlosa', 'Macrozona', 'Anio', 'SemanaEstadistica', 'TrimestreAprox']].head()

## 4) Transformaciones avanzadas (vectorización, broadcasting, pivot, reshape, chunking)

In [ ]:
df_enriched['CostoUnitarioCLP'] = np.where(
    df_enriched['NumTotal'] > 0,
    df_enriched['CostoAtencionCLP'] / df_enriched['NumTotal'],
    np.nan
)

age_matrix = df_enriched[age_cols].to_numpy(dtype=float)
means = age_matrix.mean(axis=0)
stds = age_matrix.std(axis=0)
stds = np.where(stds == 0, 1, stds)
age_z = (age_matrix - means) / stds

for i, col in enumerate(age_cols):
    df_enriched[f'{col}_z'] = age_z[:, i]

pivot_region_triage = pd.pivot_table(
    df_enriched,
    index='RegionGlosa',
    columns='PrioridadTriage',
    values='NumTotal',
    aggfunc='sum',
    fill_value=0
)

df_long_edades = df_enriched.melt(
    id_vars=['RegionGlosa', 'Anio', 'PrioridadTriage'],
    value_vars=age_cols,
    var_name='TramoEtario',
    value_name='Atenciones'
)

print('Pivot shape:', pivot_region_triage.shape)
print('Formato long shape:', df_long_edades.shape)

In [ ]:
chunk_agg = []
for chunk in pd.read_csv(RAW_PATH, chunksize=1000):
    tmp = chunk.groupby('RegionGlosa', as_index=False)['NumTotal'].sum()
    chunk_agg.append(tmp)

chunk_result = (
    pd.concat(chunk_agg, ignore_index=True)
    .groupby('RegionGlosa', as_index=False)['NumTotal']
    .sum()
    .sort_values('NumTotal', ascending=False)
)

full_result = (
    df_raw.groupby('RegionGlosa', as_index=False)['NumTotal']
    .sum()
    .sort_values('NumTotal', ascending=False)
)

check_chunk = chunk_result.merge(full_result, on='RegionGlosa', suffixes=('_chunk', '_full'))
check_chunk['Diff'] = check_chunk['NumTotal_chunk'] - check_chunk['NumTotal_full']

print('Diferencia máxima chunk vs full:', check_chunk['Diff'].abs().max())
check_chunk.head()

## 5) Verificación de integridad y validación de esquema

In [ ]:
validation_report = []

def add_check(nombre: str, condicion: bool, detalle: str = ''):
    validation_report.append({
        'check': nombre,
        'resultado': 'OK' if condicion else 'ERROR',
        'detalle': detalle
    })

add_check(
    'SemanaEstadistica en [1,53]',
    df_enriched['SemanaEstadistica'].between(1, 53).all(),
    f"Fuera de rango: {(~df_enriched['SemanaEstadistica'].between(1,53)).sum()}"
)

valid_triage = {'C1', 'C2', 'C3', 'C4', 'C5', 'ND'}
add_check(
    'PrioridadTriage válida',
    df_enriched['PrioridadTriage'].isin(valid_triage).all(),
    f"Inválidos: {(~df_enriched['PrioridadTriage'].isin(valid_triage)).sum()}"
)

valid_sexo = {'F', 'M', 'ND'}
add_check(
    'SexoPaciente válido',
    df_enriched['SexoPaciente'].isin(valid_sexo).all(),
    f"Inválidos: {(~df_enriched['SexoPaciente'].isin(valid_sexo)).sum()}"
)

add_check(
    'NumTotal = suma tramos',
    (df_enriched['NumTotal'] == df_enriched[age_cols].sum(axis=1)).all(),
    'Debe ser 0 inconsistencias luego de limpieza'
)

lat_ok = df_enriched['Latitud'].between(-56, -17)
lon_ok = df_enriched['Longitud'].between(-76, -66)
add_check(
    'Coordenadas en rango esperado Chile',
    bool((lat_ok & lon_ok).all()),
    f"Fuera de rango: {(~(lat_ok & lon_ok)).sum()}"
)

report_df = pd.DataFrame(validation_report)
report_df

## 6) Reporte y visualizaciones

In [ ]:
kpis = {
    'registros_finales': len(df_enriched),
    'atenciones_totales': int(df_enriched['NumTotal'].sum()),
    'costo_total_clp': float(df_enriched['CostoAtencionCLP'].sum()),
    'porcentaje_total_corregido': float(df_enriched['FlagTotalCorregido'].mean() * 100),
    'causa_mas_frecuente': df_enriched.groupby('Causa')['NumTotal'].sum().idxmax(),
}

kpis

In [ ]:
causas_top = (
    df_enriched.groupby('Causa', as_index=False)['NumTotal']
    .sum()
    .sort_values('NumTotal', ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
plt.barh(causas_top['Causa'], causas_top['NumTotal'])
plt.title('Top 10 causas de urgencia por atenciones')
plt.xlabel('Atenciones')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'top10_causas.png', dpi=120)
plt.show()

In [ ]:
df_enriched['Mes'] = df_enriched['FechaAtencion'].dt.to_period('M').astype('string')
serie_mensual = (
    df_enriched.dropna(subset=['Mes'])
    .groupby('Mes', as_index=False)['NumTotal']
    .sum()
)

plt.figure(figsize=(11, 4))
plt.plot(serie_mensual['Mes'], serie_mensual['NumTotal'], marker='o')
plt.title('Evolución mensual de atenciones')
plt.xlabel('Mes')
plt.ylabel('Atenciones')
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'evolucion_mensual_atenciones.png', dpi=120)
plt.show()

In [ ]:
costo_triage = (
    df_enriched.groupby('PrioridadTriage', as_index=False)['CostoAtencionCLP']
    .mean()
    .sort_values('PrioridadTriage')
)

plt.figure(figsize=(7,4))
plt.bar(costo_triage['PrioridadTriage'], costo_triage['CostoAtencionCLP'])
plt.title('Costo promedio por prioridad de triage')
plt.xlabel('Triage')
plt.ylabel('CLP promedio')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'costo_promedio_triage.png', dpi=120)
plt.show()

In [ ]:
df_enriched.to_csv(PROCESSED_PATH, index=False)
report_df.to_csv(OUTPUTS_DIR / 'reporte_validacion.csv', index=False)
pd.DataFrame([kpis]).to_csv(OUTPUTS_DIR / 'kpis_principales.csv', index=False)

processed_hash = sha256_file(PROCESSED_PATH)
print('Dataset procesado guardado en:', PROCESSED_PATH)
print('SHA256 procesado:', processed_hash)

## 7) Entorno reproducible (ítem de rúbrica)

Este notebook deja trazabilidad del entorno y crea un `requirements.txt` base.

In [ ]:
import sys
print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)

requirements = [
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"matplotlib=={plt.matplotlib.__version__}"
]

with open(PROJECT_ROOT / 'requirements.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(requirements) + '\n')

print('requirements.txt generado')


## 8) Conclusiones y recomendaciones

- Se implementó un flujo **completo y reproducible**: carga, limpieza, validación, transformación y reporte.
- Se corrigieron inconsistencias críticas (duplicados, formatos, categorías y coherencia de conteos).
- Se incorporaron técnicas de escala (chunking) y eficiencia (vectorización/broadcasting).
- Recomendación: en una siguiente iteración, formalizar reglas de esquema con `pandera` y automatizar el pipeline en scripts de `src/`.